# Install dependencies

In [1]:
!pip -q install sentence-transformers py_vncorenlp faiss-cpu rank-bm25

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 24.4 MB/s eta 0:00:00


# Import libraries and configure paths

In [2]:
import os
import json
import pickle
import shutil
import unicodedata
from typing import List, Dict, Any, Tuple

import numpy as np
import torch
import faiss
import py_vncorenlp

from tqdm.auto import tqdm
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForTokenClassification
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [3]:
VNCORENLP_DIR = os.path.abspath("/content/VnCoreNLP")
ARTIFACT_DIR = "/content/news_faiss_demo"
INDEX_PATH = os.path.join(ARTIFACT_DIR, "news.index")
METADATA_PATH = os.path.join(ARTIFACT_DIR, "metadata.jsonl")
BM25_PATH = os.path.join(ARTIFACT_DIR, "bm25.pkl")
CONFIG_PATH = os.path.join(ARTIFACT_DIR, "config.json")

DATASET_NAME = "phucdev/PhoNER_COVID19"
EMBEDDING_MODEL_NAME = "dangvantuan/vietnamese-embedding"

# Đổi path này thành thư mục model PhoBERT NER fine-tuned của bạn
NER_MODEL_PATH = "/content/drive/MyDrive/Mini Project NLP/PhoBERT"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("DEVICE:", DEVICE)
print("ARTIFACT_DIR:", ARTIFACT_DIR)

DEVICE: cuda
ARTIFACT_DIR: /content/news_faiss_demo


# Initialize Vietnamese word segmenter

In [4]:
if not os.path.exists(VNCORENLP_DIR):
    os.makedirs(VNCORENLP_DIR, exist_ok=True)

if not os.path.exists(os.path.join(VNCORENLP_DIR, "VnCoreNLP-1.2.jar")):
    py_vncorenlp.download_model(save_dir=VNCORENLP_DIR)

rdrsegmenter = py_vncorenlp.VnCoreNLP(
    annotators=["wseg"],
    save_dir=VNCORENLP_DIR
)

def word_segment_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text.strip())
    if not text:
        return ""
    segmented_list = rdrsegmenter.word_segment(text)
    return " ".join(segmented_list)

sample = "Lịch sử dịch bệnh ở Hà Nội được ghi nhận như thế nào?"
print(word_segment_text(sample))

Lịch_sử dịch_bệnh ở Hà_Nội được ghi_nhận như_thế_nào ?


# Load and normalize PhoNER_COVID19 dataset

In [5]:
dataset = load_dataset(DATASET_NAME, 'word')

print(dataset)
print("Train columns:", dataset["train"].column_names)
print("Features:", dataset["train"].features)

README.md:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

word/train-00000-of-00001.parquet:   0%|          | 0.00/366k [00:00<?, ?B/s]

word/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

word/test-00000-of-00001.parquet:   0%|          | 0.00/247k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5027 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 5027
    })
    validation: Dataset({
        features: ['words', 'tags'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 3000
    })
})
Train columns: ['words', 'tags']
Features: {'words': List(Value('string')), 'tags': List(Value('string'))}


In [6]:
def preprocess_word_list(word_list: List[str]) -> List[str]:
    return [unicodedata.normalize("NFC", str(w)) for w in word_list]

clean_dataset = dataset.map(
    lambda x: {"words": preprocess_word_list(x["words"])},
    num_proc=2
)

print(clean_dataset)

Map (num_proc=2):   0%|          | 0/5027 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['words', 'tags'],
        num_rows: 5027
    })
    validation: Dataset({
        features: ['words', 'tags'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['words', 'tags'],
        num_rows: 3000
    })
})


# Merge dataset splits into one retrieval corpus

In [7]:
splits = []

for split_name in ["train", "validation", "test"]:
    if split_name in clean_dataset:
        splits.append(clean_dataset[split_name])

corpus_dataset = concatenate_datasets(splits)

print("Total corpus size:", len(corpus_dataset))
print(corpus_dataset[0])

Total corpus size: 10027
{'words': ['Đồng_thời', ',', 'bệnh_viện', 'tiếp_tục', 'thực_hiện', 'các', 'biện_pháp', 'phòng_chống', 'dịch_bệnh', 'COVID', '-', '19', 'theo', 'hướng_dẫn', 'của', 'Bộ', 'Y_tế', '.'], 'tags': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'O']}


In [8]:
LABEL_COL = "tags"

print("LABEL_COL:", LABEL_COL)
print("Columns:", corpus_dataset.column_names)
print("Example tags:", corpus_dataset[0][LABEL_COL])

LABEL_COL: tags
Columns: ['words', 'tags']
Example tags: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-ORGANIZATION', 'I-ORGANIZATION', 'O']


# Extract entities from BIO tags

In [9]:
def extract_entities_from_bio(words, tags):
    entities = []
    current_tokens = []
    current_type = None
    start_idx = None

    def flush(end_idx):
        nonlocal current_tokens, current_type, start_idx

        if current_tokens:
            segmented_text = " ".join(current_tokens)
            display_text = segmented_text.replace("_", " ")

            entities.append({
                "text": unicodedata.normalize("NFC", display_text),
                "segmented_text": unicodedata.normalize("NFC", segmented_text),
                "type": current_type,
                "start_word": start_idx,
                "end_word": end_idx
            })

        current_tokens = []
        current_type = None
        start_idx = None

    for i, (word, tag) in enumerate(zip(words, tags)):
        if tag == "O" or tag is None:
            flush(i)
            continue

        if "-" not in tag:
            flush(i)
            continue

        prefix, ent_type = tag.split("-", 1)

        if prefix == "B":
            flush(i)
            current_tokens = [word]
            current_type = ent_type
            start_idx = i

        elif prefix == "I":
            if current_tokens and current_type == ent_type:
                current_tokens.append(word)
            else:
                flush(i)
                current_tokens = [word]
                current_type = ent_type
                start_idx = i

        else:
            flush(i)

    flush(len(words))
    return entities

In [10]:
words = corpus_dataset[3]["words"]
tags = corpus_dataset[3][LABEL_COL]

print("Segmented text:")
print(" ".join(words))

print("\nTags:")
print(tags)

print("\nEntities:")
print(extract_entities_from_bio(words, tags))

Segmented text:
Bà này khi trở về quá_cảnh Doha ( Qatar ) , đáp xuống Tân_Sơn_Nhất sáng 2/3 cùng 75 hành_khách , trong đó có 55 người nước_ngoài .

Tags:
['O', 'O', 'O', 'O', 'O', 'O', 'B-LOCATION', 'O', 'B-LOCATION', 'O', 'O', 'O', 'O', 'B-LOCATION', 'O', 'B-DATE', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Entities:
[{'text': 'Doha', 'segmented_text': 'Doha', 'type': 'LOCATION', 'start_word': 6, 'end_word': 7}, {'text': 'Qatar', 'segmented_text': 'Qatar', 'type': 'LOCATION', 'start_word': 8, 'end_word': 9}, {'text': 'Tân Sơn Nhất', 'segmented_text': 'Tân_Sơn_Nhất', 'type': 'LOCATION', 'start_word': 13, 'end_word': 14}, {'text': '2/3', 'segmented_text': '2/3', 'type': 'DATE', 'start_word': 15, 'end_word': 16}]


# Build document metadata

In [11]:
def join_segmented_words(words):
    words = [unicodedata.normalize("NFC", str(w)) for w in words]
    return " ".join(words)


def segmented_to_display_text(segmented_text):
    return segmented_text.replace("_", " ")


def build_document_from_example(example, doc_id):
    words = example["words"]
    tags = example[LABEL_COL]

    segmented_text = join_segmented_words(words)
    display_text = segmented_to_display_text(segmented_text)

    entities = extract_entities_from_bio(words, tags)

    return {
        "id": f"news_{doc_id}",
        "display_text": display_text,
        "segmented_text": segmented_text,
        "entities": entities,
        "entity_texts": sorted(list(set(e["text"] for e in entities))),
        "entity_segmented_texts": sorted(list(set(e["segmented_text"] for e in entities))),
        "entity_types": sorted(list(set(e["type"] for e in entities))),
    }

In [12]:
metadata = []

for i, example in enumerate(tqdm(corpus_dataset, desc="Building metadata")):
    metadata.append(build_document_from_example(example, i))

print("Num docs:", len(metadata))
print(json.dumps(metadata[0], ensure_ascii=False, indent=2))

Building metadata:   0%|          | 0/10027 [00:00<?, ?it/s]

Num docs: 10027
{
  "id": "news_0",
  "display_text": "Đồng thời , bệnh viện tiếp tục thực hiện các biện pháp phòng chống dịch bệnh COVID - 19 theo hướng dẫn của Bộ Y tế .",
  "segmented_text": "Đồng_thời , bệnh_viện tiếp_tục thực_hiện các biện_pháp phòng_chống dịch_bệnh COVID - 19 theo hướng_dẫn của Bộ Y_tế .",
  "entities": [
    {
      "text": "Bộ Y tế",
      "segmented_text": "Bộ Y_tế",
      "type": "ORGANIZATION",
      "start_word": 15,
      "end_word": 17
    }
  ],
  "entity_texts": [
    "Bộ Y tế"
  ],
  "entity_segmented_texts": [
    "Bộ Y_tế"
  ],
  "entity_types": [
    "ORGANIZATION"
  ]
}


# Build BM25 index

In [16]:
tokenized_corpus = [
    doc["segmented_text"].split()
    for doc in metadata
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 corpus size:", len(tokenized_corpus))
print("Example tokens:")
print(tokenized_corpus[0][:30])

BM25 corpus size: 10027
Example tokens:
['Đồng_thời', ',', 'bệnh_viện', 'tiếp_tục', 'thực_hiện', 'các', 'biện_pháp', 'phòng_chống', 'dịch_bệnh', 'COVID', '-', '19', 'theo', 'hướng_dẫn', 'của', 'Bộ', 'Y_tế', '.']


# Encode corpus with Vietnamese embedding model

In [13]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

texts_for_embedding = [doc["segmented_text"] for doc in metadata]

print("Num texts:", len(texts_for_embedding))
print("Example segmented text:")
print(texts_for_embedding[0])

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

...

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Num texts: 10027
Example segmented text:
Đồng_thời , bệnh_viện tiếp_tục thực_hiện các biện_pháp phòng_chống dịch_bệnh COVID - 19 theo hướng_dẫn của Bộ Y_tế .


In [14]:
BATCH_SIZE = 64

embeddings = embedder.encode(
    texts_for_embedding,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

print("Embeddings shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Embeddings shape: (10027, 768)
Embedding dtype: float32


# Build FAISS vector index

In [15]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("FAISS dimension:", dimension)
print("FAISS ntotal:", index.ntotal)

FAISS dimension: 768
FAISS ntotal: 10027


# Save retrieval artifacts

In [17]:
faiss.write_index(index, INDEX_PATH)

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    for doc in metadata:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

with open(BM25_PATH, "wb") as f:
    pickle.dump(bm25, f)

config = {
    "dataset_name": DATASET_NAME,
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "dimension": int(dimension),
    "num_docs": len(metadata),
    "index_type": "faiss.IndexFlatIP",
    "embedding_normalized": True,
    "bm25": "rank_bm25.BM25Okapi",
    "label_col": LABEL_COL,
    "note": "PhoNER_COVID19 words are already word-segmented. Corpus uses joined segmented_text directly."
}

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("Saved:")
print(INDEX_PATH)
print(METADATA_PATH)
print(BM25_PATH)
print(CONFIG_PATH)

Saved:
/content/news_faiss_demo/news.index
/content/news_faiss_demo/metadata.jsonl
/content/news_faiss_demo/bm25.pkl
/content/news_faiss_demo/config.json


In [18]:
ZIP_PATH = "/content/news_faiss_demo.zip"

if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

shutil.make_archive(
    base_name=ZIP_PATH.replace(".zip", ""),
    format="zip",
    root_dir=ARTIFACT_DIR
)

print("ZIP_PATH:", ZIP_PATH)

ZIP_PATH: /content/news_faiss_demo.zip


In [32]:
from google.colab import files
files.download(ZIP_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Load retrieval artifacts for demo

In [19]:
def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows


index = faiss.read_index(INDEX_PATH)

with open(BM25_PATH, "rb") as f:
    bm25 = pickle.load(f)

metadata = load_jsonl(METADATA_PATH)

print("Loaded FAISS ntotal:", index.ntotal)
print("Loaded metadata:", len(metadata))
print(metadata[0])

Loaded FAISS ntotal: 10027
Loaded metadata: 10027
{'id': 'news_0', 'display_text': 'Đồng thời , bệnh viện tiếp tục thực hiện các biện pháp phòng chống dịch bệnh COVID - 19 theo hướng dẫn của Bộ Y tế .', 'segmented_text': 'Đồng_thời , bệnh_viện tiếp_tục thực_hiện các biện_pháp phòng_chống dịch_bệnh COVID - 19 theo hướng_dẫn của Bộ Y_tế .', 'entities': [{'text': 'Bộ Y tế', 'segmented_text': 'Bộ Y_tế', 'type': 'ORGANIZATION', 'start_word': 15, 'end_word': 17}], 'entity_texts': ['Bộ Y tế'], 'entity_segmented_texts': ['Bộ Y_tế'], 'entity_types': ['ORGANIZATION']}


# Load fine-tuned PhoBERT NER model

In [20]:
ner_tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2", use_fast=False)

ner_model = AutoModelForTokenClassification.from_pretrained(NER_MODEL_PATH)
ner_model.to(DEVICE)
ner_model.eval()

id2label = ner_model.config.id2label

print("NER labels:", id2label)

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

NER labels: {0: 'B-AGE', 1: 'B-DATE', 2: 'B-GENDER', 3: 'B-JOB', 4: 'B-LOCATION', 5: 'B-NAME', 6: 'B-ORGANIZATION', 7: 'B-PATIENT_ID', 8: 'B-SYMPTOM_AND_DISEASE', 9: 'B-TRANSPORTATION', 10: 'I-AGE', 11: 'I-DATE', 12: 'I-JOB', 13: 'I-LOCATION', 14: 'I-NAME', 15: 'I-ORGANIZATION', 16: 'I-PATIENT_ID', 17: 'I-SYMPTOM_AND_DISEASE', 18: 'I-TRANSPORTATION', 19: 'O'}


# Predict entities from user query

In [23]:
def predict_ner_entities(text, max_length=256):
    """
    Predict NER cho input người dùng.

    Không dùng encoded.word_ids() vì PhoBERT tokenizer có thể không hỗ trợ fast tokenizer.
    Ta tự tokenize từng word đã được word-segment và tự lưu mapping token -> word.
    """

    segmented_text = word_segment_text(text)
    words = segmented_text.split()

    if not words:
        return segmented_text, [], []

    tokens = []
    token_to_word = []

    for word_idx, word in enumerate(words):
        sub_tokens = ner_tokenizer.tokenize(word)

        if len(sub_tokens) == 0:
            continue

        for sub_token in sub_tokens:
            tokens.append(sub_token)
            token_to_word.append(word_idx)

    # Chừa chỗ cho <s> và </s>
    max_token_len = max_length - 2

    tokens = tokens[:max_token_len]
    token_to_word = token_to_word[:max_token_len]

    input_ids = ner_tokenizer.convert_tokens_to_ids(tokens)

    cls_token_id = ner_tokenizer.cls_token_id
    sep_token_id = ner_tokenizer.sep_token_id
    pad_token_id = ner_tokenizer.pad_token_id

    if cls_token_id is None:
        cls_token_id = ner_tokenizer.convert_tokens_to_ids("<s>")

    if sep_token_id is None:
        sep_token_id = ner_tokenizer.convert_tokens_to_ids("</s>")

    input_ids = [cls_token_id] + input_ids + [sep_token_id]
    attention_mask = [1] * len(input_ids)

    input_ids_tensor = torch.tensor([input_ids], dtype=torch.long).to(DEVICE)
    attention_mask_tensor = torch.tensor([attention_mask], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        outputs = ner_model(
            input_ids=input_ids_tensor,
            attention_mask=attention_mask_tensor
        )

    pred_ids = outputs.logits.argmax(dim=-1)[0].detach().cpu().tolist()

    # Bỏ prediction của <s> và </s>
    pred_ids_without_special = pred_ids[1:-1]

    word_level_labels = []
    seen_word_ids = set()

    for token_idx, word_idx in enumerate(token_to_word):
        if word_idx in seen_word_ids:
            continue

        seen_word_ids.add(word_idx)

        label_id = int(pred_ids_without_special[token_idx])
        label = id2label[label_id]

        word_level_labels.append({
            "word": words[word_idx],
            "label": label
        })

    pred_words = [x["word"] for x in word_level_labels]
    pred_tags = [x["label"] for x in word_level_labels]

    entities = extract_entities_from_bio(pred_words, pred_tags)

    return segmented_text, word_level_labels, entities

In [24]:
test_query = "Bệnh nhân 129 ở Hà Nội từng nhập cảnh qua sân bay Nội Bài."

segmented_query, word_labels, entities = predict_ner_entities(test_query)

print("Segmented query:")
print(segmented_query)

print("\nWord labels:")
for item in word_labels:
    print(item)

print("\nEntities:")
for ent in entities:
    print(ent)

Segmented query:
Bệnh_nhân 129 ở Hà_Nội từng nhập_cảnh qua sân_bay Nội_Bài .

Word labels:
{'word': 'Bệnh_nhân', 'label': 'O'}
{'word': '129', 'label': 'B-PATIENT_ID'}
{'word': 'ở', 'label': 'O'}
{'word': 'Hà_Nội', 'label': 'B-LOCATION'}
{'word': 'từng', 'label': 'O'}
{'word': 'nhập_cảnh', 'label': 'O'}
{'word': 'qua', 'label': 'O'}
{'word': 'sân_bay', 'label': 'B-LOCATION'}
{'word': 'Nội_Bài', 'label': 'I-LOCATION'}
{'word': '.', 'label': 'O'}

Entities:
{'text': '129', 'segmented_text': '129', 'type': 'PATIENT_ID', 'start_word': 1, 'end_word': 2}
{'text': 'Hà Nội', 'segmented_text': 'Hà_Nội', 'type': 'LOCATION', 'start_word': 3, 'end_word': 4}
{'text': 'sân bay Nội Bài', 'segmented_text': 'sân_bay Nội_Bài', 'type': 'LOCATION', 'start_word': 7, 'end_word': 9}


# Define retrieval methods

In [25]:
def min_max_normalize(scores):
    scores = np.asarray(scores, dtype=np.float32)

    if len(scores) == 0:
        return scores

    min_score = scores.min()
    max_score = scores.max()

    if max_score - min_score < 1e-8:
        return np.ones_like(scores)

    return (scores - min_score) / (max_score - min_score)


def vector_search(query, top_k=5):
    segmented_query = word_segment_text(query)

    query_embedding = embedder.encode(
        [segmented_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx < 0:
            continue

        doc = metadata[int(idx)]

        results.append({
            "doc_id": doc["id"],
            "score": float(score),
            "method": "vector",
            "display_text": doc["display_text"],
            "segmented_text": doc["segmented_text"],
            "entities": doc["entities"]
        })

    return results


def bm25_search(query, top_k=5):
    segmented_query = word_segment_text(query)
    query_tokens = segmented_query.split()

    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        doc = metadata[int(idx)]

        results.append({
            "doc_id": doc["id"],
            "score": float(scores[idx]),
            "method": "bm25",
            "display_text": doc["display_text"],
            "segmented_text": doc["segmented_text"],
            "entities": doc["entities"]
        })

    return results


def hybrid_search(query, top_k=5, candidate_k=50, alpha=0.6):
    segmented_query = word_segment_text(query)

    query_embedding = embedder.encode(
        [segmented_query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    vector_scores, vector_indices = index.search(query_embedding, candidate_k)

    query_tokens = segmented_query.split()
    bm25_scores_all = bm25.get_scores(query_tokens)

    candidate_indices = set(int(i) for i in vector_indices[0] if i >= 0)

    bm25_top_indices = np.argsort(bm25_scores_all)[::-1][:candidate_k]
    candidate_indices.update(int(i) for i in bm25_top_indices)

    candidate_indices = list(candidate_indices)

    vector_score_map = {
        int(idx): float(score)
        for score, idx in zip(vector_scores[0], vector_indices[0])
        if idx >= 0
    }

    vector_scores_raw = np.array(
        [vector_score_map.get(idx, 0.0) for idx in candidate_indices],
        dtype=np.float32
    )

    bm25_scores_raw = np.array(
        [bm25_scores_all[idx] for idx in candidate_indices],
        dtype=np.float32
    )

    vector_scores_norm = min_max_normalize(vector_scores_raw)
    bm25_scores_norm = min_max_normalize(bm25_scores_raw)

    final_scores = alpha * vector_scores_norm + (1.0 - alpha) * bm25_scores_norm

    ranked = sorted(
        zip(candidate_indices, final_scores, vector_scores_raw, bm25_scores_raw),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []

    for idx, final_score, vector_score, bm25_score in ranked:
        doc = metadata[int(idx)]

        results.append({
            "doc_id": doc["id"],
            "score": float(final_score),
            "vector_score": float(vector_score),
            "bm25_score": float(bm25_score),
            "method": "hybrid",
            "display_text": doc["display_text"],
            "segmented_text": doc["segmented_text"],
            "entities": doc["entities"]
        })

    return results

# Build entity-aware search query

In [26]:
def build_entity_query(entity_text, original_user_text=None):
    entity_text = unicodedata.normalize("NFC", entity_text.strip())

    if original_user_text:
        original_user_text = unicodedata.normalize("NFC", original_user_text.strip())
        return f"{entity_text}. Ngữ cảnh: {original_user_text}"

    return entity_text


def search(query, method="hybrid", top_k=5, alpha=0.6):
    method = method.lower().strip()

    if method == "vector":
        return vector_search(query, top_k=top_k)

    if method == "bm25":
        return bm25_search(query, top_k=top_k)

    if method == "hybrid":
        return hybrid_search(
            query=query,
            top_k=top_k,
            candidate_k=max(50, top_k * 10),
            alpha=alpha
        )

    raise ValueError("method phải là một trong: vector, bm25, hybrid")

# Format and print search results

In [27]:
def print_entities(entities):
    if not entities:
        print("Không tìm thấy entity nào.")
        return

    for i, ent in enumerate(entities, start=1):
        print(f"[{i}] {ent['text']} | segmented={ent['segmented_text']} | type={ent['type']}")


def print_results(results):
    if not results:
        print("Không có kết quả.")
        return

    for rank, item in enumerate(results, start=1):
        print("=" * 100)
        print(f"Rank: {rank}")
        print(f"Doc ID: {item['doc_id']}")
        print(f"Method: {item['method']}")
        print(f"Score: {item['score']:.4f}")

        if item["method"] == "hybrid":
            print(f"Vector score: {item['vector_score']:.4f}")
            print(f"BM25 score: {item['bm25_score']:.4f}")

        print("-" * 100)
        print("Display text:")
        print(item["display_text"])

        print("-" * 100)
        print("Segmented text:")
        print(item["segmented_text"])

        if item.get("entities"):
            shown_entities = [
                f"{e['text']}({e['type']})"
                for e in item["entities"][:10]
            ]
            print("-" * 100)
            print("Entities:", ", ".join(shown_entities))

# Demo

In [29]:
def run_cli_demo():
    print("NEWS ENTITY RETRIEVAL DEMO")
    print("Nhập 'exit' để thoát.")
    print("-" * 100)

    while True:
        user_text = input("\nNhập câu truy vấn: ").strip()

        if user_text.lower() in ["exit", "quit", "q"]:
            print("Thoát demo.")
            break

        if not user_text:
            print("Vui lòng nhập câu không rỗng.")
            continue

        segmented_query, word_labels, entities = predict_ner_entities(user_text)

        print("\nSegmented query:")
        print(segmented_query)

        print("\nEntities nhận diện được:")
        print_entities(entities)

        if entities:
            selected = input(
                "\nChọn số entity để search, hoặc Enter để search toàn bộ câu: "
            ).strip()

            if selected:
                try:
                    selected_idx = int(selected) - 1
                    selected_entity = entities[selected_idx]
                    final_query = build_entity_query(
                        selected_entity["text"],
                        original_user_text=user_text
                    )
                except Exception:
                    print("Lựa chọn không hợp lệ. Search bằng toàn bộ câu.")
                    final_query = user_text
            else:
                final_query = user_text
        else:
            final_query = user_text

        method = input(
            "\nChọn method [bm25/vector/hybrid], mặc định hybrid: "
        ).strip().lower()

        if method == "":
            method = "hybrid"

        if method not in ["bm25", "vector", "hybrid"]:
            print("Method không hợp lệ. Dùng hybrid.")
            method = "hybrid"

        top_k_str = input("Nhập top_k, mặc định 5: ").strip()

        try:
            top_k = int(top_k_str) if top_k_str else 5
        except Exception:
            top_k = 5

        alpha = 0.6

        if method == "hybrid":
            alpha_str = input(
                "Nhập alpha cho hybrid, 0.0=BM25, 1.0=vector, mặc định 0.6: "
            ).strip()

            try:
                alpha = float(alpha_str) if alpha_str else 0.6
                alpha = max(0.0, min(1.0, alpha))
            except Exception:
                alpha = 0.6

        print("\nFinal query:")
        print(final_query)

        results = search(
            query=final_query,
            method=method,
            top_k=top_k,
            alpha=alpha
        )

        print("\nKết quả:")
        print_results(results)

In [31]:
run_cli_demo()

NEWS ENTITY RETRIEVAL DEMO
Nhập 'exit' để thoát.
----------------------------------------------------------------------------------------------------

Nhập câu truy vấn: Bệnh nhân 129 ở Hà Nội từng nhập cảnh qua sân bay Nội Bài.

Segmented query:
Bệnh_nhân 129 ở Hà_Nội từng nhập_cảnh qua sân_bay Nội_Bài .

Entities nhận diện được:
[1] 129 | segmented=129 | type=PATIENT_ID
[2] Hà Nội | segmented=Hà_Nội | type=LOCATION
[3] sân bay Nội Bài | segmented=sân_bay Nội_Bài | type=LOCATION

Chọn số entity để search, hoặc Enter để search toàn bộ câu: 3

Chọn method [bm25/vector/hybrid], mặc định hybrid: vector
Nhập top_k, mặc định 5: 10

Final query:
sân bay Nội Bài. Ngữ cảnh: Bệnh nhân 129 ở Hà Nội từng nhập cảnh qua sân bay Nội Bài.

Kết quả:
Rank: 1
Doc ID: news_10020
Method: vector
Score: 0.6108
----------------------------------------------------------------------------------------------------
Display text:
Trước đó ngày 10 - 3 , hệ thống giám sát bệnh truyền nhiễm Việt Nam đã ghi nhận một t

In [28]:
query = "Bệnh nhân 129 ở Hà Nội từng nhập cảnh qua sân bay Nội Bài."

segmented_query, word_labels, entities = predict_ner_entities(query)

print("Segmented query:")
print(segmented_query)

print("\nEntities:")
print_entities(entities)

if entities:
    selected_entity = entities[0]["text"]
    final_query = build_entity_query(selected_entity, query)
else:
    final_query = query

results = search(
    query=final_query,
    method="hybrid",
    top_k=5,
    alpha=0.6
)

print("\nFinal query:")
print(final_query)

print("\nResults:")
print_results(results)

Segmented query:
Bệnh_nhân 129 ở Hà_Nội từng nhập_cảnh qua sân_bay Nội_Bài .

Entities:
[1] 129 | segmented=129 | type=PATIENT_ID
[2] Hà Nội | segmented=Hà_Nội | type=LOCATION
[3] sân bay Nội Bài | segmented=sân_bay Nội_Bài | type=LOCATION

Final query:
129. Ngữ cảnh: Bệnh nhân 129 ở Hà Nội từng nhập cảnh qua sân bay Nội Bài.

Results:
Rank: 1
Doc ID: news_749
Method: hybrid
Score: 0.9605
Vector score: 0.5525
BM25 score: 31.9100
----------------------------------------------------------------------------------------------------
Display text:
" Bệnh nhân 129 " , nam , 20 tuổi ở Nghĩa Tân , Hà Nội , du học sinh tại Anh nhập cảnh về Nội Bài ngày 20/3 trên chuyến bay VN 54 .
----------------------------------------------------------------------------------------------------
Segmented text:
" Bệnh_nhân 129 " , nam , 20 tuổi ở Nghĩa Tân , Hà_Nội , du_học_sinh tại Anh nhập_cảnh về Nội_Bài ngày 20/3 trên chuyến bay VN 54 .
-----------------------------------------------------------------------